# Chapter 2 · Hands-On Agent Frameworks: CrewAI and Tool Use
**Mastering Agentic AI** — Manning Publications


## 2.0 · Setup

See `appendix_a_api_keys.md` for credential setup.

> CrewAI defaults to OpenAI but can be configured for Anthropic. Both keys shown in the appendix.

In [ ]:
# !pip install anthropic crewai python-dotenv
import os, json
from anthropic import Anthropic
from dotenv import load_dotenv
load_dotenv()
assert os.getenv("ANTHROPIC_API_KEY"), "See appendix_a_api_keys.md"
client = Anthropic()
MODEL = "claude-opus-4-5"
print("Ready")

## 2.1 · The Three Core Tools

In [ ]:
NUTRITION_DB = {
    "apple":          {"calories": 95,  "protein_g": 0.5,  "carbs_g": 25, "fat_g": 0.3, "fibre_g": 4.4},
    "chicken breast": {"calories": 165, "protein_g": 31.0, "carbs_g": 0,  "fat_g": 3.6, "fibre_g": 0.0},
    "oats":           {"calories": 150, "protein_g": 5.0,  "carbs_g": 27, "fat_g": 2.5, "fibre_g": 4.0},
    "salmon":         {"calories": 208, "protein_g": 28.0, "carbs_g": 0,  "fat_g": 10.0,"fibre_g": 0.0},
}
meal_log = []

def lookup_nutrition(food_item: str) -> str:
    "Fetch macro-nutrient data for a food item per 100g serving."
    key = food_item.strip().lower()
    data = NUTRITION_DB.get(key)
    if data: return json.dumps({"food": key, **data})
    matches = [k for k in NUTRITION_DB if key in k]
    if matches: return json.dumps({"food": matches[0], "note": "closest match", **NUTRITION_DB[matches[0]]})
    return json.dumps({"error": f"'{food_item}' not in database"})

def log_meal(food_item: str, grams: float) -> str:
    "Log a consumed food item. Scales nutrients from 100g reference to the actual portion."
    n = json.loads(lookup_nutrition(food_item))
    if "error" in n: return json.dumps(n)
    scale = grams / 100
    entry = {k: round(v * scale, 1) if isinstance(v, float) else v for k, v in n.items()}
    entry["grams"] = grams
    meal_log.append(entry)
    return json.dumps({"logged": entry, "total_entries": len(meal_log)})

def get_daily_summary() -> str:
    "Return total macro-nutrient intake across all logged meals today."
    if not meal_log: return json.dumps({"message": "No meals logged yet."})
    totals = {"calories": 0, "protein_g": 0, "carbs_g": 0, "fat_g": 0}
    for e in meal_log:
        for k in totals: totals[k] = round(totals[k] + e.get(k, 0), 1)
    return json.dumps({"meals_logged": len(meal_log), "totals": totals})

print(lookup_nutrition("salmon"))

## 2.2 · Tool Schemas for the API

The model sees **only** the name, description, and schema. The docstring IS the user interface.

In [ ]:
TOOLS = [
    {
        "name": "lookup_nutrition",
        "description": (
            "Fetch macro-nutrient data (calories, protein, carbs, fat, fibre) "
            "for a single raw food item per 100g. Use for individual ingredients only."
        ),
        "input_schema": {"type": "object",
            "properties": {"food_item": {"type": "string", "description": "Common food name"}},
            "required": ["food_item"]}
    },
    {
        "name": "log_meal",
        "description": "Record that the user ate a food item in grams.",
        "input_schema": {"type": "object",
            "properties": {"food_item": {"type": "string"}, "grams": {"type": "number"}},
            "required": ["food_item", "grams"]}
    },
    {
        "name": "get_daily_summary",
        "description": "Return total macro-nutrient intake from all logged meals today.",
        "input_schema": {"type": "object", "properties": {}, "required": []}
    },
]
TOOL_FNS = {"lookup_nutrition": lookup_nutrition, "log_meal": log_meal, "get_daily_summary": get_daily_summary}
print(f"{len(TOOLS)} tools registered")

## 2.3 · The Raw Tool-Use Loop

In [ ]:
def run_tool_loop(user_message: str) -> str:
    """The fundamental tool-use pattern. You are the executor."""
    messages = [{"role": "user", "content": user_message}]
    while True:
        response = client.messages.create(
            model=MODEL, max_tokens=1024, system="You are an AI Diet Coach.",
            tools=TOOLS, messages=messages)
        messages.append({"role": "assistant", "content": response.content})
        if response.stop_reason == "end_turn":
            return "".join(b.text for b in response.content if b.type == "text")
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                fn = TOOL_FNS.get(block.name)
                result = fn(**block.input) if fn else json.dumps({"error": "unknown"})
                print(f"  → {block.name}({block.input})")
                tool_results.append({"type": "tool_result", "tool_use_id": block.id, "content": result})
        messages.append({"role": "user", "content": tool_results})

print(run_tool_loop("How many calories in 150g of salmon?"))

## 2.4 · DietCoachAgent Wrapper

In [ ]:
class DietCoachAgent:
    "Object-oriented wrapper. Exposes clean chat() interface; maintains history internally."
    def __init__(self):
        self.client = Anthropic()
        self.history = []
        self.system = "You are an AI Diet Coach with access to nutrition tools. Be concise."
    def chat(self, user_message: str) -> str:
        self.history.append({"role": "user", "content": user_message})
        while True:
            r = self.client.messages.create(model=MODEL, max_tokens=1024,
                system=self.system, tools=TOOLS, messages=self.history)
            self.history.append({"role": "assistant", "content": r.content})
            if r.stop_reason == "end_turn":
                return "".join(b.text for b in r.content if b.type == "text")
            results = []
            for b in r.content:
                if b.type == "tool_use":
                    fn = TOOL_FNS.get(b.name)
                    result = fn(**b.input) if fn else json.dumps({"error": "unknown"})
                    results.append({"type": "tool_result", "tool_use_id": b.id, "content": result})
            self.history.append({"role": "user", "content": results})

coach = DietCoachAgent()
print(coach.chat("What is the protein content of brown rice?"))
print(coach.chat("Log 200g of it for my lunch."))
print(coach.chat("What are my totals today?"))

## 2.5 · Summary

| Concept | What we built |
|---|---|
| Tool schemas | 3 tools with docstrings as the LLM user interface |
| Raw loop | `while stop_reason != end_turn` — the core tool pattern |
| DietCoachAgent | Class with persistent history |

**Next:** Chapter 3 — prompting, Agent Skills, and context engineering.

---
*Mastering Agentic AI · Manning Publications*